# 全量数据提取（唯一程序 v2）

自动取**最新已收盘交易日**（北京时间），输出单个 Excel `supermind_full_YYYYMMDD.xlsx`，9 个 sheet：

- **market** 全市场股票+22个常用指数快照，含行业(同花顺1/2/3级、申万1/2级)与概念标签
- **sentiment** 总量情绪指标汇总（成交额、涨跌家数、各档位家数、连板、新高、两融合计…）
- **theme_index** 专题板块指数点位（微盘/次新/新高/连板/涨停/龙头/成交前十/情绪…）
- **theme_members** 上述每个板块的成份股逐只展开（成份股数 = 家数，不需历史）
- **mtss** 融资融券（数据滞后约1个交易日，sheet 内 mtss_date 标注实际日期）
- **stocks_meta / valuation / indexes_all / indexes_meta** 元数据与全量指数

**用法**：菜单 运行 → 运行所有单元格（约 25 分钟，主要耗在全量指数那格）。
想指定日期就改第一格的 `TARGET_DATE`。

注：估值每天晚上才入库，白天跑 valuation 会是空表；概念库按 T-1 更新。

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# 全量数据提取（唯一程序 v2）—— 最新已收盘交易日，北京时间
#   market / sentiment / theme_index / theme_members / mtss
#   / stocks_meta / valuation / indexes_all / indexes_meta
# 输出: 单个 Excel  supermind_full_YYYYMMDD.xlsx
# 用法: 菜单 运行 → 运行所有单元格
# ============================================================
TARGET_DATE = None   # None = 自动取最新已收盘交易日; 或手动 '2026-08-04'

from mindgo_api import *
import pandas as pd

now = pd.Timestamp.now()                      # 研究环境服务器 = 北京时间
_all_days = pd.to_datetime(pd.Series(list(get_all_trade_days())))
if TARGET_DATE is None:
    past = _all_days[_all_days <= now.normalize()].reset_index(drop=True)
    last = past.iloc[-1]
    if last.date() == now.date() and (now.hour * 60 + now.minute) < (15 * 60 + 5):
        last = past.iloc[-2]
        print('今天尚未收盘(北京 %s), 自动取上一交易日' % now.strftime('%H:%M'))
    TARGET_DATE = last.strftime('%Y-%m-%d')
elif not (_all_days == pd.Timestamp(TARGET_DATE)).any():
    raise ValueError('%s 不是交易日' % TARGET_DATE)

DS = TARGET_DATE.replace('-', '')
OUT_XLSX = 'supermind_full_%s.xlsx' % DS
_prev = _all_days[_all_days < pd.Timestamp(TARGET_DATE)]
PREV_DATE = _prev.iloc[-1].strftime('%Y-%m-%d')          # 上一交易日(两融/概念滞后用)

# 常用指数（含你要的微盘股、近端次新股）
INDEX_CODES = [
    '000001.SH', '399001.SZ', '399006.SZ', '000688.SH', '000698.SH',
    '899050.BJ', '000300.SH', '000016.SH', '000010.SH', '000905.SH',
    '000852.SH', '000906.SH', '000903.SH', '000985.CSI', '399303.SZ',
    '399005.SZ', '399330.SZ', '399673.SZ', '000015.SH', '000922.CSI',
    '883418.TI',   # 微盘股
    '883907.TI',   # 近端次新股
]

# 专题/情绪板块指数：取点位 + 成份股名单（成份股数 = 家数，无需历史）
THEME_INDEXES = [
    ('883957.TI', '同花顺全A(沪深京)'), ('883421.TI', '同花顺全A(沪深)'),
    ('883404.TI', '同花顺情绪指数'),
    ('883418.TI', '微盘股'),
    ('883907.TI', '近端次新股'), ('883974.TI', '远端次新股'),
    ('883965.TI', '破发次新股'), ('885598.TI', '新股与次新股'),
    ('885905.TI', '注册制次新股'), ('885907.TI', '科创次新股'),
    ('883911.TI', '创历史新高'), ('883408.TI', '近期新高'), ('883416.TI', '百日新高'),
    ('883958.TI', '昨日连板'), ('883988.TI', '昨日非ST连板'),
    ('883900.TI', '昨日涨停表现'), ('883986.TI', '昨日非ST涨停表现'),
    ('883423.TI', '沪深主板昨日涨停'), ('883424.TI', '创业科创板昨日涨停'),
    ('883902.TI', '昨日成交前十'),
    ('883908.TI', '沪股通成交前十'), ('883912.TI', '深股通成交前十'),
    ('883917.TI', '行业龙头'),
    ('885338.TI', '融资融券'), ('883969.TI', '融资盘高持仓'),
]

FIELDS = ['open', 'high', 'low', 'close', 'prev_close', 'avg_price',
          'quote_rate', 'amp_rate', 'volume', 'turnover', 'turnover_rate',
          'high_limit', 'low_limit', 'factor', 'is_paused', 'is_st']
IDX_FIELDS = ['open', 'high', 'low', 'close', 'prev_close', 'avg_price',
              'quote_rate', 'amp_rate', 'volume', 'turnover']
MTSS_FIELDS = ['fin_value', 'fin_buy_value', 'fin_refund_value',
               'sec_value', 'fin_sec_value']
print('目标交易日: %s (上一交易日 %s) -> %s' % (TARGET_DATE, PREV_DATE, OUT_XLSX))


In [ ]:
# ---- 1) 全市场股票 + 常用指数 日线快照 ----
import gc, time
t0 = time.time()
sec_stock = get_all_securities('stock', TARGET_DATE)
stock_codes = list(sec_stock.index)
print('%s 在市股票 %d 只' % (TARGET_DATE, len(stock_codes)))

def fetch(codes, kind, fields):
    panel = get_price(codes, TARGET_DATE, TARGET_DATE, '1d',
                      fields, False, None, 0, is_panel=True)
    data = {f: panel[f].iloc[0].reindex(codes) for f in fields}
    df = pd.DataFrame(data, index=pd.Index(codes, name='symbol')).reset_index()
    df.insert(0, 'date', TARGET_DATE)
    df.insert(2, 'asset_type', kind)
    return df

df_market = pd.concat([fetch(stock_codes, 'stock', FIELDS),
                       fetch(INDEX_CODES, 'index', FIELDS)], ignore_index=True)
name_map = sec_stock['display_name'].to_dict() if 'display_name' in sec_stock.columns else {}
exch_map = sec_stock['exchange'].to_dict() if 'exchange' in sec_stock.columns else {}
for code in INDEX_CODES:
    try:
        info = get_security_info(code)
        name_map[code] = info.display_name
        exch_map[code] = info.exchange
    except Exception:
        name_map.setdefault(code, code)
        exch_map.setdefault(code, '')
df_market['display_name'] = df_market['symbol'].map(name_map)
df_market['exchange'] = df_market['symbol'].map(exch_map)
n_miss = int(df_market['close'].isna().sum())
df_market = df_market[df_market['close'].notna()].copy()
df_market = df_market[['date', 'symbol', 'display_name', 'asset_type', 'exchange'] + FIELDS]
df_market = df_market.sort_values(['asset_type', 'symbol']).reset_index(drop=True)
print('market: %d 行(剔除当日无行情 %d), %.0fs' % (len(df_market), n_miss, time.time() - t0))
gc.collect()


In [ ]:
# ---- 2) 行业 + 概念标签（原始标签, 不做聚合分析）----
import gc, time
t0 = time.time()

# 2a 行业: 逐只查(约1分钟), 拿同花顺一/二/三级 + 申万一/二级 的 ID, 再统一映射成名称
rows = []
for c in stock_codes:
    try:
        si = get_symbol_industry(c)
        rows.append((c, si.industryid1, si.industryid2, si.industryid3,
                     si.s_industryid1, si.s_industryid2))
    except Exception:
        rows.append((c, None, None, None, None, None))
ind = pd.DataFrame(rows, columns=['symbol', 'ths_l1_id', 'ths_l2_id', 'ths_l3_id',
                                  'sw_l1_id', 'sw_l2_id'])
ids = pd.unique(ind[['ths_l1_id', 'ths_l2_id', 'ths_l3_id',
                     'sw_l1_id', 'sw_l2_id']].values.ravel())
id2name = {}
for i in ids:
    if i is None or (isinstance(i, float) and pd.isna(i)):
        continue
    try:
        o = get_industry_info(i)
        id2name[i] = None if o is None else o.industry_name
    except Exception:
        id2name[i] = None
for src, dst in [('ths_l1_id', 'industry_ths_l1'), ('ths_l2_id', 'industry_ths_l2'),
                 ('ths_l3_id', 'industry_ths_l3'), ('sw_l1_id', 'industry_sw_l1'),
                 ('sw_l2_id', 'industry_sw_l2')]:
    ind[dst] = ind[src].map(id2name)
print('行业: %d 只, %d 个行业码, %.0fs' % (len(ind), len(id2name), time.time() - t0))

# 2b 概念: 整表按日期取(概念库按 T-1 更新, 自动回退找最近有数据的日期)
concept_date, cdf = None, pd.DataFrame()
for d in [DS, PREV_DATE.replace('-', '')]:
    try:
        r = run_query(query(concept_classification)
                      .filter(concept_classification.date == d).limit(9000))
        if len(r):
            concept_date, cdf = d, r
            break
    except Exception as e:
        print('概念查询 %s 失败: %r' % (d, e))
if len(cdf):
    cdf = cdf.rename(columns={'concept_classification_symbol': 'symbol',
                              'concept_classification_concept': 'concepts'})
    ind = ind.merge(cdf[['symbol', 'concepts']], on='symbol', how='left')
    print('概念: 日期 %s, 覆盖 %d 只' % (concept_date, int(cdf['symbol'].nunique())))
else:
    ind['concepts'] = None
    print('!! 概念表当日与前一日均为空')

tag_cols = ['symbol', 'industry_ths_l1', 'industry_ths_l2', 'industry_ths_l3',
            'industry_sw_l1', 'industry_sw_l2', 'concepts']
df_tags = ind[tag_cols]
df_market = df_market.merge(df_tags, on='symbol', how='left')
print('market 加标签后: %d 行 x %d 列' % df_market.shape)
del rows, ind, cdf
gc.collect()


In [ ]:
# ---- 3) 专题/情绪板块指数: 点位 + 成份股名单 ----
import gc, time
t0 = time.time()
codes = [c for c, _ in THEME_INDEXES]
panel = get_price(codes, TARGET_DATE, TARGET_DATE, '1d',
                  IDX_FIELDS, False, None, 0, is_panel=True)
data = {f: panel[f].iloc[0].reindex(codes) for f in IDX_FIELDS}
df_theme = pd.DataFrame(data, index=pd.Index(codes, name='symbol')).reset_index()
df_theme.insert(0, 'date', TARGET_DATE)
df_theme.insert(2, 'theme_name', [n for _, n in THEME_INDEXES])
del panel

mem_rows, counts = [], {}
q = df_market.set_index('symbol')
for code, nm in THEME_INDEXES:
    try:
        members = get_index_stocks(code, TARGET_DATE) or []
    except Exception as e:
        print('  %s %s 成份股取失败: %r' % (code, nm, e))
        members = []
    counts[code] = len(members)
    for s in members:
        r = q.loc[s] if s in q.index else None
        mem_rows.append({
            'date': TARGET_DATE, 'theme_code': code, 'theme_name': nm, 'symbol': s,
            'display_name': None if r is None else r['display_name'],
            'quote_rate': None if r is None else r['quote_rate'],
            'close': None if r is None else r['close'],
            'turnover': None if r is None else r['turnover'],
            'industry_ths_l2': None if r is None else r.get('industry_ths_l2'),
            'concepts': None if r is None else r.get('concepts'),
        })
df_theme['member_count'] = df_theme['symbol'].map(counts)
df_theme_members = pd.DataFrame(mem_rows)
print('theme_index: %d 个指数; theme_members: %d 行, %.0fs'
      % (len(df_theme), len(df_theme_members), time.time() - t0))
for c, n in THEME_INDEXES:
    print('  %-11s %-16s 成份 %5s' % (c, n, counts.get(c)))
del mem_rows, q
gc.collect()


In [ ]:
# ---- 4) 融资融券（数据滞后, 取最近可用交易日）----
import gc, time
t0 = time.time()
start = _all_days[_all_days <= pd.Timestamp(TARGET_DATE)].iloc[-6].strftime('%Y-%m-%d')
parts = []
CH = 500
for i in range(0, len(stock_codes), CH):
    ch = stock_codes[i:i + CH]
    try:
        d = get_mtss(ch, start, TARGET_DATE, MTSS_FIELDS)
        for sym, sub in (d or {}).items():
            if sub is None or not len(sub):
                continue
            last = sub.iloc[[-1]].copy()
            last.insert(0, 'symbol', sym)
            last.insert(1, 'mtss_date', sub.index[-1].strftime('%Y-%m-%d'))
            parts.append(last)
    except Exception as e:
        print('  两融块 %d 失败: %r' % (i // CH, e))
    if (i // CH) % 4 == 0:
        print('  两融 %d/%d, %.0fs' % (min(i + CH, len(stock_codes)), len(stock_codes),
                                       time.time() - t0))
df_mtss = (pd.concat(parts, ignore_index=True) if parts else pd.DataFrame())
if len(df_mtss):
    df_mtss.insert(0, 'query_date', TARGET_DATE)
    print('mtss: %d 只, 数据日期 %s (相对 %s 滞后), %.0fs'
          % (len(df_mtss), sorted(df_mtss['mtss_date'].unique())[-1], TARGET_DATE,
             time.time() - t0))
else:
    print('!! 两融为空')
del parts
gc.collect()


In [ ]:
# ---- 5) 总量情绪指标汇总 ----
import gc
s = df_market[df_market.asset_type == 'stock']
tr = s[s.is_paused == 0]                     # 当日正常交易的
lim = tr[tr.high_limit > 0]                  # 上市首日无涨跌停限制, 排除
up_lim = lim[lim.close >= lim.high_limit - 1e-6]
dn_lim = lim[lim.close <= lim.low_limit + 1e-6]
yest_up = set(df_theme_members.loc[df_theme_members.theme_code == '883900.TI', 'symbol'])
lianban_today = up_lim[up_lim['symbol'].isin(yest_up)]
tm = df_theme_members
def cnt(code):
    return int((tm.theme_code == code).sum())
big10 = tr[tr.quote_rate > 10]
M = [
    ('两市总成交额(沪深京)', tr['turnover'].sum(), '元'),
    ('两市总成交额(沪深)', tr[tr.exchange != '北交所']['turnover'].sum(), '元'),
    ('上涨家数', int((tr.quote_rate > 0).sum()), '家'),
    ('下跌家数', int((tr.quote_rate < 0).sum()), '家'),
    ('平盘家数', int((tr.quote_rate == 0).sum()), '家'),
    ('涨停家数', len(up_lim), '家'),
    ('跌停家数', len(dn_lim), '家'),
    ('今日连板家数(今日涨停∩昨日涨停)', len(lianban_today), '家'),
    ('昨日连板家数(883958成份)', cnt('883958.TI'), '家'),
    ('涨幅>10%家数', int((tr.quote_rate > 10).sum()), '家'),
    ('涨幅>5%家数', int((tr.quote_rate > 5).sum()), '家'),
    ('涨幅>3%家数', int((tr.quote_rate > 3).sum()), '家'),
    ('跌幅>3%家数', int((tr.quote_rate < -3).sum()), '家'),
    ('跌幅>5%家数', int((tr.quote_rate < -5).sum()), '家'),
    ('涨幅>10%成交额合计', big10['turnover'].sum(), '元'),
    ('平均股价', tr['close'].mean(), '元'),
    ('全A股价中位数', tr['close'].median(), '元'),
    ('全A涨幅中位数', tr['quote_rate'].median(), '%'),
    ('创历史新高家数(883911成份)', cnt('883911.TI'), '家'),
    ('近期新高家数(883408成份)', cnt('883408.TI'), '家'),
    ('百日新高家数(883416成份)', cnt('883416.TI'), '家'),
    ('近端次新股家数(883907成份)', cnt('883907.TI'), '家'),
    ('微盘股成份数(883418)', cnt('883418.TI'), '家'),
    ('行业龙头成份数(883917)', cnt('883917.TI'), '家'),
    ('正常交易股票数', len(tr), '家'),
    ('停牌家数', int(s['is_paused'].sum()), '家'),
    ('ST家数', int(s['is_st'].sum()), '家'),
]
if len(df_mtss):
    md = sorted(df_mtss['mtss_date'].unique())[-1]
    sub = df_mtss[df_mtss.mtss_date == md]
    M += [('融资余额合计(%s)' % md, sub['fin_value'].sum(), '元'),
          ('融券余额合计(%s)' % md, sub['sec_value'].sum(), '元'),
          ('两融余额合计(%s)' % md, sub['fin_sec_value'].sum(), '元'),
          ('融资买入额合计(%s)' % md, sub['fin_buy_value'].sum(), '元')]
for c, n in [('883404.TI', '同花顺情绪指数'), ('883957.TI', '同花顺全A(沪深京)'),
             ('883418.TI', '微盘股指数'), ('883907.TI', '近端次新股指数'),
             ('883911.TI', '创历史新高指数'), ('883917.TI', '行业龙头指数'),
             ('883902.TI', '昨日成交前十指数')]:
    r = df_theme.loc[df_theme.symbol == c, 'close']
    if len(r):
        M.append((n + '(点位)', float(r.iloc[0]), '点'))
df_sentiment = pd.DataFrame(M, columns=['指标', '数值', '单位'])
df_sentiment.insert(0, 'date', TARGET_DATE)
pd.set_option('display.float_format', lambda v: '%.2f' % v)
print(df_sentiment.to_string(index=False))
gc.collect()


In [ ]:
# ---- 6) 全量指数(约2.3万)当日行情 + 元数据（最耗时, 15-20分钟）----
import gc, time
t0 = time.time()
sec_index = get_all_securities('index', TARGET_DATE)
idx_codes = list(sec_index.index)
df_indexes_meta = sec_index.reset_index().rename(columns={'index': 'symbol'})
iname = sec_index['display_name'].to_dict() if 'display_name' in sec_index.columns else {}
print('全量指数 %d 个' % len(idx_codes))
parts, failed = [], []
CH = 2000
for i in range(0, len(idx_codes), CH):
    ch = idx_codes[i:i + CH]
    try:
        panel = get_price(ch, TARGET_DATE, TARGET_DATE, '1d',
                          IDX_FIELDS, False, None, 0, is_panel=True)
        data = {f: panel[f].iloc[0].reindex(ch) for f in IDX_FIELDS}
        parts.append(pd.DataFrame(data, index=pd.Index(ch, name='symbol')).reset_index())
        del panel
    except Exception as e:
        print('!! 块 %d 失败: %r, 改200一组' % (i // CH, e))
        for j in range(0, len(ch), 200):
            sub = ch[j:j + 200]
            try:
                panel = get_price(sub, TARGET_DATE, TARGET_DATE, '1d',
                                  IDX_FIELDS, False, None, 0, is_panel=True)
                data = {f: panel[f].iloc[0].reindex(sub) for f in IDX_FIELDS}
                parts.append(pd.DataFrame(data, index=pd.Index(sub, name='symbol')).reset_index())
                del panel
            except Exception:
                failed.extend(sub)
    print('指数进度 %d/%d, %.0fs' % (min(i + CH, len(idx_codes)), len(idx_codes),
                                    time.time() - t0))
    gc.collect()
df_indexes_all = pd.concat(parts, ignore_index=True)
df_indexes_all.insert(0, 'date', TARGET_DATE)
df_indexes_all['display_name'] = df_indexes_all['symbol'].map(iname)
df_indexes_all['has_quote'] = df_indexes_all['close'].notna()
print('indexes_all: %d 行(有行情 %d), 失败 %d, %.0fs'
      % (len(df_indexes_all), int(df_indexes_all['has_quote'].sum()), len(failed),
         time.time() - t0))
del parts, sec_index
gc.collect()


In [ ]:
# ---- 7) 股票元数据 + 估值 ----
import gc, time
t0 = time.time()
df_stocks_meta = sec_stock.reset_index().rename(columns={'index': 'symbol'})
val_parts, val_err = [], None
try:
    for i in range(0, len(stock_codes), 400):
        ch = stock_codes[i:i + 400]
        val_parts.append(get_fundamentals(
            query(valuation).filter(valuation.symbol.in_(ch)), date=DS))
except Exception as e:
    val_err = e
    print('!! 估值查询失败: %r' % (e,))
df_val = (pd.concat(val_parts, ignore_index=True)
          if val_parts and val_err is None else pd.DataFrame())
if len(df_val) == 0:
    print('!!!! 估值为空 —— 每日估值当晚入库, 晚些或次日重跑本格即可补上')
else:
    if 'date' not in df_val.columns:
        df_val.insert(0, 'date', TARGET_DATE)
    print('valuation: %d 行 x %d 列, %.0fs' % (df_val.shape[0], df_val.shape[1],
                                              time.time() - t0))
print('stocks_meta: %d 行 x %d 列' % df_stocks_meta.shape)
del val_parts
gc.collect()


In [ ]:
# ---- 8) 合并写入单个 Excel + 校验 ----
import gc, time
t0 = time.time()
try:
    import xlsxwriter  # noqa: F401
    engine = 'xlsxwriter'
except ImportError:
    engine = 'openpyxl'
sheets = [('market', df_market), ('sentiment', df_sentiment),
          ('theme_index', df_theme), ('theme_members', df_theme_members),
          ('mtss', df_mtss), ('stocks_meta', df_stocks_meta),
          ('valuation', df_val), ('indexes_all', df_indexes_all),
          ('indexes_meta', df_indexes_meta)]
with pd.ExcelWriter(OUT_XLSX, engine=engine) as w:
    for nm, df in sheets:
        df.to_excel(w, sheet_name=nm, index=False)
        print('  sheet %-14s %6d 行 x %2d 列' % (nm, df.shape[0], df.shape[1]))
print('已保存 %s (%s), %.0fs' % (OUT_XLSX, engine, time.time() - t0))

stk = df_market[df_market.asset_type == 'stock']
idx = df_market[df_market.asset_type == 'index']
checks = [
    ('股票行数>=5000', len(stk) >= 5000, len(stk)),
    ('常用指数数==%d' % len(INDEX_CODES), len(idx) == len(INDEX_CODES), len(idx)),
    ('日期唯一且正确', list(df_market['date'].unique()) == [TARGET_DATE], ''),
    ('high>=low', bool(((stk['high'] - stk['low']).dropna() >= 0).all()), ''),
    ('行业标签覆盖>95%', stk['industry_ths_l1'].notna().mean() > 0.95,
     '%.1f%%' % (stk['industry_ths_l1'].notna().mean() * 100)),
    ('概念标签覆盖>90%', stk['concepts'].notna().mean() > 0.90,
     '%.1f%%' % (stk['concepts'].notna().mean() * 100)),
    ('专题成份股非空', len(df_theme_members) > 0, len(df_theme_members)),
    ('两融非空', len(df_mtss) > 0, len(df_mtss)),
    ('全量指数>=20000', len(df_indexes_all) >= 20000, len(df_indexes_all)),
    ('估值非空(空=未入库)', len(df_val) > 0, len(df_val)),
]
fails = 0
for nm, ok, d in checks:
    print('  [%s] %s %s' % ('PASS' if ok else 'FAIL', nm, d))
    fails += (not ok)
r = idx.loc[idx.symbol == '000001.SH', 'close']
print('上证指数 close:', float(r.iloc[0]) if len(r) else 'N/A')
print('ALL DONE' if fails == 0 else '有 %d 项未通过(见上)' % fails)
del df_market, df_sentiment, df_theme, df_theme_members, df_mtss
del df_stocks_meta, df_val, df_indexes_all, df_indexes_meta, sec_stock
gc.collect()
print('内存已释放')
if fails:
    raise RuntimeError('%d validation checks failed' % fails)
